# Exploratory Data Analysis (EDA)
Pure exploratory profiling of the raw LTA Bus OD, Bus Stops, NEA Rainfall, and HDB Carpark Availability datasets. All data cleaning, quarantine splitting, and canonical exports are handled separately in `eda_cleaning_updates.py`.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("userfiles") / "CA1_Datasets"
if not DATA_DIR.exists() and Path("CA1_Datasets").exists():
    DATA_DIR = Path("CA1_Datasets")

carpark2609 = pd.read_csv(
    DATA_DIR / "carpark_20260901_20260930.csv",
    dtype={
        "carpark_number": "string",
        "lot_type": "string"
    }
)

bus_stops2609 = pd.read_csv(
    DATA_DIR / "bus_stops.csv",
    dtype={
        "BusStopCode": "string",
        "RoadName": "string",
        "Description": "string"
    }
)

# Load all available monthly Bus OD CSVs (e.g. 202608, 202609, 202610)
bus_od_dtype = {
    "YEAR_MONTH": "string",
    "DAY_TYPE": "string",
    "PT_TYPE": "string",
    "ORIGIN_PT_CODE": "string",
    "DESTINATION_PT_CODE": "string"
}

seen_od_names = set()
bus_od_files = []
for p in sorted(DATA_DIR.rglob("origin_destination_bus_*.csv")):
    if p.name not in seen_od_names:
        seen_od_names.add(p.name)
        bus_od_files.append(p)

od_frames = []
for fpath in bus_od_files:
    df_month = pd.read_csv(fpath, dtype=bus_od_dtype)
    print(f"Loaded {fpath.name}: {len(df_month):,} rows (months: {df_month['YEAR_MONTH'].unique().tolist()})")
    od_frames.append(df_month)

od_bus2609 = pd.concat(od_frames, ignore_index=True)
del od_frames
print(f"Combined Bus OD total rows: {len(od_bus2609):,}")

rainfall_2609 = pd.read_csv(
    DATA_DIR / "rainfall_20260901_20260930.csv",
    dtype={
        "station_id": "string",
        "station_name": "string"
    }
)

datasets = {
    "carpark": carpark2609,
    "bus_stops": bus_stops2609,
    "bus_od": od_bus2609,
    "rainfall": rainfall_2609
}

In [4]:
for name, df in datasets.items():

    print(f"\n{'=' * 60}")
    print(f"Dataset {name.upper()}")
    print(f"{'=' * 60}")

    print(f"Rows: {len(df):,}")
    print(f"Columns: {len(df.columns)}")

    print("\nData types:")
    print(df.dtypes)

    print("\nMissing values:")
    print(df.isna().sum())

    print("\nBlank string values:")
    print(
        df.select_dtypes(include=["object", "string"])
        .apply(lambda col: col.str.strip().eq("").sum())
    )

    print("\nDuplicate rows:")
    print(df.duplicated().sum())


Dataset CARPARK
Rows: 1,745,692
Columns: 6

Data types:
snapshot_time              object
carpark_number     string[python]
lot_type           string[python]
lots_available              int64
total_lots                  int64
update_datetime            object
dtype: object

Missing values:
snapshot_time      0
carpark_number     0
lot_type           0
lots_available     0
total_lots         0
update_datetime    0
dtype: int64

Blank string values:
snapshot_time      0
carpark_number     0
lot_type           0
update_datetime    0
dtype: int64

Duplicate rows:
0

Dataset BUS_STOPS
Rows: 5,209
Columns: 5

Data types:
BusStopCode    string[python]
RoadName       string[python]
Description    string[python]
Latitude              float64
Longitude             float64
dtype: object

Missing values:
BusStopCode    0
RoadName       0
Description    0
Latitude       0
Longitude      0
dtype: int64

Blank string values:
BusStopCode    0
RoadName       0
Description    0
dtype: int64

Duplicate 

### Check duplicate analytical keys

In [5]:
# Bus OD
bus_od_key = [
    "YEAR_MONTH",
    "DAY_TYPE",
    "TIME_PER_HOUR",
    "PT_TYPE",
    "ORIGIN_PT_CODE",
    "DESTINATION_PT_CODE"
]

# Rainfall
rainfall_key = [
    "station_id",
    "timestamp"
]

# Carpark
carpark_key = [
    "carpark_number",
    "lot_type",
    "update_datetime"
]

# Bus stops
bus_stop_key = [
    "BusStopCode"
]

key_checks = {
    "Bus OD": (od_bus2609, bus_od_key),
    "Rainfall": (rainfall_2609, rainfall_key),
    "Carpark": (carpark2609, carpark_key),
    "Bus Stops": (bus_stops2609, bus_stop_key)
}

for name, (df, keys) in key_checks.items():

    duplicate_mask = df.duplicated(
        subset=keys,
        keep=False
    )

    duplicate_records = df[duplicate_mask]

    print(f"\n{name}")
    print(f"Duplicate key rows: {len(duplicate_records):,}")
    print(
        f"Duplicate key groups: "
        f"{df.duplicated(subset=keys).sum():,}"
    )


Bus OD
Duplicate key rows: 0
Duplicate key groups: 0

Rainfall
Duplicate key rows: 0
Duplicate key groups: 0

Carpark
Duplicate key rows: 21,223
Duplicate key groups: 16,555

Bus Stops
Duplicate key rows: 0
Duplicate key groups: 0


### Check invalid timestamp formats

In [6]:
print("BUS OD")
print("YEAR_MONTH:", od_bus2609["YEAR_MONTH"].unique()[:15])
print("DAY_TYPE:", od_bus2609["DAY_TYPE"].unique())
print("TIME_PER_HOUR:", od_bus2609["TIME_PER_HOUR"].unique())
print("PT_TYPE:", od_bus2609["PT_TYPE"].unique())

print("\nCARPARK")
print("LOT_TYPE:", carpark2609["lot_type"].unique())

print("\nRAINFALL")
print("Rainfall range:")
print(rainfall_2609["rainfall_mm"].describe())

BUS OD
YEAR_MONTH: <StringArray>
['2026-09']
Length: 1, dtype: string
DAY_TYPE: <StringArray>
['WEEKDAY', 'WEEKENDS/HOLIDAY']
Length: 2, dtype: string
TIME_PER_HOUR: [19 12 10 18 11  0  8  6 17  7 21 16 23 13 20 22 14 15  5  9  1  2  4  3]
PT_TYPE: <StringArray>
['BUS']
Length: 1, dtype: string

CARPARK
LOT_TYPE: <StringArray>
['C', 'Y', 'H', 'S']
Length: 4, dtype: string

RAINFALL
Rainfall range:
count    761445.000000
mean          0.023575
std           0.317894
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          19.700000
Name: rainfall_mm, dtype: float64


In [7]:
def numeric_profile(df, column, name):

    values = pd.to_numeric(df[column], errors="coerce")

    invalid = values.isna()
    print(f"\n{name} - {column}")
    print(f"Unparseable values: {invalid.sum():,}")

    return values


# Bus OD
trips = numeric_profile(
    od_bus2609, "TOTAL_TRIPS", "Bus OD"
)

print("Negative passenger counts:", (trips < 0).sum())


# Rainfall
rain = numeric_profile(
    rainfall_2609, "rainfall_mm", "Rainfall"
)

print("Negative rainfall values:", (rain < 0).sum())


# Carpark
available = numeric_profile(
    carpark2609, "lots_available", "Carpark"
)

capacity = numeric_profile(
    carpark2609, "total_lots", "Carpark"
)

invalid_capacity = (
    (capacity <= 0) |
    (available < 0) |
    (available > capacity)
)

print(
    "Invalid parking capacity/availability records:",
    invalid_capacity.sum()
)


Bus OD - TOTAL_TRIPS
Unparseable values: 0
Negative passenger counts: 0

Rainfall - rainfall_mm
Unparseable values: 0
Negative rainfall values: 0

Carpark - lots_available
Unparseable values: 0

Carpark - total_lots
Unparseable values: 0
Invalid parking capacity/availability records: 3908


In [ ]:
timestamp_columns = {
    "Carpark snapshot": (carpark2609, "snapshot_time"),
    "Carpark source update": (carpark2609, "update_datetime"),
    "Rainfall observation": (rainfall_2609, "timestamp")
}

for name, (df, column) in timestamp_columns.items():
    parsed = pd.to_datetime(
        df[column],
        format="ISO8601",
        errors="coerce",
        utc=True,
        cache=True
    )

    print(f"\n{name}")
    print("Unparseable timestamps:", parsed.isna().sum())
    print("Earliest:", parsed.min())
    print("Latest:", parsed.max())

In [8]:
timestamp_columns = {
    "Carpark snapshot": (carpark2609, "snapshot_time"),
    "Carpark source update": (carpark2609, "update_datetime"),
    "Rainfall observation": (rainfall_2609, "timestamp")
}

for name, (df, column) in timestamp_columns.items():

    parsed = pd.to_datetime(
        df[column],
        errors="coerce",
        utc=True
    )

    print(f"\n{name}")
    print("Unparseable timestamps:", parsed.isna().sum())
    print("Earliest:", parsed.min())
    print("Latest:", parsed.max())


Carpark snapshot
Unparseable timestamps: 0
Earliest: 2026-08-31 16:00:37+00:00
Latest: 2026-09-30 15:00:36+00:00

Carpark source update
Unparseable timestamps: 0
Earliest: 2025-11-01 13:45:00+00:00
Latest: 2026-09-30 23:01:04+00:00

Rainfall observation
Unparseable timestamps: 0
Earliest: 2026-08-31 16:00:00+00:00
Latest: 2026-09-30 15:55:00+00:00


## Exploring carpark dataset issues
- Duplicates
- Invalid records
- Unique timestamp records

In [9]:
# Inspect duplicate carpark key groups
carpark_key = ["carpark_number", "lot_type", "update_datetime"]

duplicate_mask = carpark2609.duplicated(
    subset=carpark_key,
    keep=False
)

carpark_duplicates = carpark2609[duplicate_mask].copy()

# Count how many distinct values exist within each duplicate key
duplicate_summary = (
    carpark_duplicates
    .groupby(carpark_key)
    .agg(
        row_count=("snapshot_time", "size"),
        distinct_snapshots=("snapshot_time", "nunique"),
        distinct_available=("lots_available", "nunique"),
        distinct_capacity=("total_lots", "nunique")
    )
    .reset_index()
)

print("Duplicate groups:", len(duplicate_summary))
print("\nValue variation within duplicate groups:")
print(
    duplicate_summary[
        ["distinct_snapshots", "distinct_available", "distinct_capacity"]
    ].value_counts().head(20)
)

print("\nGroups with conflicting availability or capacity:")
print(
    duplicate_summary[
        (duplicate_summary["distinct_available"] > 1) |
        (duplicate_summary["distinct_capacity"] > 1)
    ].head(20)
)

Duplicate groups: 4668

Value variation within duplicate groups:
distinct_snapshots  distinct_available  distinct_capacity
2                   1                   1                    3202
3                   1                   1                     713
4                   1                   1                     357
5                   1                   1                     131
6                   1                   1                      63
7                   1                   1                      43
8                   1                   1                      29
10                  1                   1                      21
9                   1                   1                      20
12                  1                   1                      18
11                  1                   1                      12
720                 1                   1                       8
20                  1                   1                       7
16                 

In [10]:
# Break down invalid carpark records by reason
available = pd.to_numeric(carpark2609["lots_available"], errors="coerce")
capacity = pd.to_numeric(carpark2609["total_lots"], errors="coerce")

invalid_reasons = pd.DataFrame({
    "capacity_zero_or_negative": capacity <= 0,
    "availability_negative": available < 0,
    "availability_exceeds_capacity": available > capacity,
    "capacity_missing_or_unparseable": capacity.isna(),
    "availability_missing_or_unparseable": available.isna()
})

print("Invalid records by condition:")
print(invalid_reasons.sum())

invalid_mask = invalid_reasons.any(axis=1)

print("\nTotal invalid records:", invalid_mask.sum())

print("\nSample invalid records:")
display(
    carpark2609.loc[
        invalid_mask,
        [
            "snapshot_time", "carpark_number", "lot_type",
            "lots_available", "total_lots", "update_datetime"
        ]
    ].head(20)
)

Invalid records by condition:
capacity_zero_or_negative              3188
availability_negative                     0
availability_exceeds_capacity           720
capacity_missing_or_unparseable           0
availability_missing_or_unparseable       0
dtype: int64

Total invalid records: 3908

Sample invalid records:


,snapshot_time,carpark_number,lot_type,lots_available,total_lots,update_datetime
757,2026-09-01T00:00:37+08:00,PL40,C,0,0,2026-08-04T00:00:00
966,2026-09-01T00:00:37+08:00,JM33,S,0,0,2026-05-07T14:39:09
1851,2026-09-01T00:00:37+08:00,PL57,S,0,0,2026-04-16T20:27:39
1971,2026-09-01T00:00:37+08:00,A88,Y,64,46,2026-08-31T23:51:14
2201,2026-09-01T00:00:37+08:00,A33,C,0,0,2026-08-31T23:59:35
3181,2026-09-01T01:00:37+08:00,PL40,C,0,0,2026-08-04T00:00:00
3388,2026-09-01T01:00:37+08:00,JM33,S,0,0,2026-05-07T14:39:09
4268,2026-09-01T01:00:37+08:00,PL57,S,0,0,2026-04-16T20:27:39
4392,2026-09-01T01:00:37+08:00,A88,Y,64,46,2026-09-01T00:45:47
4624,2026-09-01T01:00:37+08:00,A33,C,0,0,2026-09-01T01:00:06


In [11]:
print("Carpark snapshot_time:")
print(carpark2609["snapshot_time"].head(5).to_list())

print("\nCarpark update_datetime:")
print(carpark2609["update_datetime"].head(10).to_list())

print("\nRainfall timestamp:")
print(rainfall_2609["timestamp"].head(5).to_list())

Carpark snapshot_time:
['2026-09-01T00:00:37+08:00', '2026-09-01T00:00:37+08:00', '2026-09-01T00:00:37+08:00', '2026-09-01T00:00:37+08:00', '2026-09-01T00:00:37+08:00']

Carpark update_datetime:
['2026-08-31T23:59:45', '2026-09-01T00:00:04', '2026-08-31T23:59:46', '2026-09-01T00:00:07', '2026-08-31T23:59:59', '2026-09-01T00:00:10', '2026-09-01T00:00:20', '2026-08-31T23:59:56', '2026-09-01T00:00:08', '2026-08-31T23:59:40']

Rainfall timestamp:
['2026-09-01T23:55:00+08:00', '2026-09-01T23:55:00+08:00', '2026-09-01T23:55:00+08:00', '2026-09-01T23:55:00+08:00', '2026-09-01T23:55:00+08:00']


In [ ]:
# Carpark snapshot coverage
carpark_snapshots = pd.to_datetime(
    carpark2609["snapshot_time"],
    format="ISO8601",
    errors="coerce",
    utc=True,
    cache=True
).dt.tz_convert("Asia/Singapore")

print("CARPARK SNAPSHOT COVERAGE")
print("Unique snapshots:", carpark_snapshots.nunique())
print("Earliest:", carpark_snapshots.min())
print("Latest:", carpark_snapshots.max())

print("\nSnapshots per Singapore calendar date:")
print(
    carpark_snapshots.dt.date.value_counts().sort_index().describe()
)

# Rainfall coverage
rainfall_times = pd.to_datetime(
    rainfall_2609["timestamp"],
    format="ISO8601",
    errors="coerce",
    utc=True,
    cache=True
).dt.tz_convert("Asia/Singapore")

print("\nRAINFALL COVERAGE")
print("Unique stations:", rainfall_2609["station_id"].nunique())
print("Unique timestamps:", rainfall_times.nunique())
print("Rows per station:")
print(rainfall_2609.groupby("station_id").size().describe())

print("\nObservations per station per date:")
rainfall_daily_eda = (
    rainfall_2609.assign(local_date=rainfall_times.dt.date)
    .groupby(["station_id", "local_date"])
    .agg(
        observation_count=("timestamp", "size"),
        unique_timestamps=("timestamp", "nunique")
    )
)

print(rainfall_daily_eda.describe())
print(
    "\nStation-days with fewer than 288 unique timestamps:",
    (rainfall_daily_eda["unique_timestamps"] < 288).sum()
)

## Profiling Observation Keys, Slot Alignment, and Referential Integrity

## Solving the issues
- Duplicates
- Invalid records
- Unique timestamp records

### Duplicates

In [ ]:
# Count unique five-minute timestamps per station per Singapore date
station_daily_coverage = (
    rainfall_2609.assign(local_date=rainfall_times.dt.date)
    .groupby(["station_id", "local_date"])["timestamp"]
    .nunique()
    .rename("observed_slots")
    .reset_index()
)

# Build the complete station-date grid, including station-days with zero rows
all_stations = rainfall_2609["station_id"].dropna().unique()
all_dates = pd.date_range("2026-09-01", "2026-09-30").date

full_grid = pd.MultiIndex.from_product(
    [all_stations, all_dates],
    names=["station_id", "local_date"]
).to_frame(index=False)

station_daily_coverage = full_grid.merge(
    station_daily_coverage,
    on=["station_id", "local_date"],
    how="left"
)

station_daily_coverage["observed_slots"] = (
    station_daily_coverage["observed_slots"]
    .fillna(0)
    .astype(int)
)

station_daily_coverage["coverage_pct"] = (
    station_daily_coverage["observed_slots"] / 288 * 100
)

station_daily_coverage["eligible_for_ranking"] = (
    station_daily_coverage["observed_slots"] >= 260
)

print("Total expected station-days:", len(station_daily_coverage))
print(
    "Eligible station-days (>=260 slots):",
    station_daily_coverage["eligible_for_ranking"].sum()
)
print(
    "Partial station-days below 90% (<260 slots):",
    (~station_daily_coverage["eligible_for_ranking"]).sum()
)

print("\nCoverage distribution:")
print(
    station_daily_coverage["observed_slots"]
    .value_counts()
    .sort_index()
)

print("\nStation-days with lowest coverage:")
display(
    station_daily_coverage
    .sort_values("observed_slots")
    .head(20)
)

### Rainfall timestamp slot alignment

In [ ]:
# RAINFALL SLOT ALIGNMENT PROFILE
print("Minute distribution:")
print(rainfall_times.dt.minute.value_counts().sort_index())

print("\nSecond distribution:")
print(rainfall_times.dt.second.value_counts().sort_index())

valid_minutes = {0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55}
invalid_alignment = (
    ~rainfall_times.dt.minute.isin(valid_minutes) |
    (rainfall_times.dt.second != 0)
)
print("\nMisaligned observations:", invalid_alignment.sum())

## Bus OD Validation & Bus Stop Referential Integrity

In [ ]:
# BUS OD PROFILE BY MONTH, DAY TYPE, AND HOUR
bus_od_required = {
    "YEAR_MONTH",
    "DAY_TYPE",
    "TIME_PER_HOUR",
    "PT_TYPE",
    "ORIGIN_PT_CODE",
    "DESTINATION_PT_CODE",
    "TOTAL_TRIPS"
}

print("Missing required columns:", bus_od_required - set(od_bus2609.columns))

print("\nRows per month:")
print(od_bus2609["YEAR_MONTH"].value_counts(dropna=False).sort_index())

print("\nDay types:")
print(od_bus2609["DAY_TYPE"].value_counts(dropna=False))

print("\nPublic transport types:")
print(od_bus2609["PT_TYPE"].value_counts(dropna=False))

print("\nInvalid hours:", (~od_bus2609["TIME_PER_HOUR"].between(0, 23)).sum())
print("Negative trips:", (od_bus2609["TOTAL_TRIPS"] < 0).sum())

valid_month = od_bus2609["YEAR_MONTH"].str.fullmatch(r"\d{4}-(0[1-9]|1[0-2])", na=False)
print("Invalid year_month values:", (~valid_month).sum())

In [ ]:
# BUS STOP REFERENTIAL INTEGRITY
stop_ids = set(bus_stops2609["BusStopCode"].dropna())

origin_exists = od_bus2609["ORIGIN_PT_CODE"].isin(stop_ids)
destination_exists = od_bus2609["DESTINATION_PT_CODE"].isin(stop_ids)

print("Origin codes not found in bus_stops.csv:", (~origin_exists).sum())
print("Destination codes not found in bus_stops.csv:", (~destination_exists).sum())

print("\nTop unmatched origins:")
print(od_bus2609.loc[~origin_exists, "ORIGIN_PT_CODE"].value_counts().head(20))

print("\nTop unmatched destinations:")
print(od_bus2609.loc[~destination_exists, "DESTINATION_PT_CODE"].value_counts().head(20))

### Prepare rainfall daily coverage

In [ ]:
# Carpark source freshness profile on valid carpark rows
valid_cp_mask = (
    (carpark2609["total_lots"] > 0)
    & (carpark2609["lots_available"] >= 0)
    & (carpark2609["lots_available"] <= carpark2609["total_lots"])
)

snapshot_sg = carpark_snapshots.loc[valid_cp_mask].dt.tz_localize(None)
update_dt = pd.to_datetime(
    carpark2609.loc[valid_cp_mask, "update_datetime"],
    format="ISO8601",
    errors="coerce",
    cache=True
)
update_datetime_age = snapshot_sg - update_dt

print(update_datetime_age.describe())

print("\nSource updates older than:")
for hours in [1, 6, 12, 24, 72, 168]:
    count = (update_datetime_age > pd.Timedelta(hours=hours)).sum()
    print(f"{hours:>3} hours: {count:,}")

## BUS OD VALIDATION

In [ ]:
# LOT TYPE C PROFILE (on valid carpark rows)
carpark_c = carpark2609.loc[
    valid_cp_mask & (carpark2609["lot_type"] == "C")
].copy()

print("Valid Lot type C records:", len(carpark_c))
print("Unique carparks:", carpark_c["carpark_number"].nunique())

print("\nCapacity summary:")
print(carpark_c["total_lots"].describe())

print("\nAvailability summary:")
print(carpark_c["lots_available"].describe())

occupancy_ratio = (
    carpark_c["total_lots"] - carpark_c["lots_available"]
) / carpark_c["total_lots"]

print("\nOccupancy ratio summary:")
print(occupancy_ratio.describe())
print("Occupancy ratio outside [0, 1]:", (~occupancy_ratio.between(0, 1)).sum())